# notebook-SlopGPT

Petit notebook pour generer une image avec le modele deja entraine.

Important: ce notebook ne lance aucun entrainement. Il charge le checkpoint existant dans `outputs/quantum_sim_fashion_mnist_16x16_angles.npz`, puis appelle le generateur deja present dans `generate_quantum_sim_sample.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

%matplotlib inline

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "generate_quantum_sim_sample.py").exists():
    NOTEBOOK_DIR = Path("SlopGPT-QuantumTrain").resolve()

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from generate_quantum_sim_sample import generate_result, match_prompt
from train_quantum_sim_fashion_mnist_16 import CHECKPOINT_PATH, FASHION_CLASSES

print("Notebook dir:", NOTEBOOK_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Missing trained checkpoint: {CHECKPOINT_PATH}. "
        "Run the training script once first, or restore the outputs folder."
    )

## Choose a prompt

Examples: `sneaker`, `cool futuristic sneakers`, `small boot`, `tall boots`, `bag`, `dress`, `trouser`, `coat`, `shirt`.

In [ ]:
prompt = "cool futuristic sneakers"
shots = 128
seed = 42
variation = 0.08
latent_scale = 1.0
candidates = 4

label, class_name = match_prompt(prompt)
print(f"Prompt: {prompt}")
print(f"Matched class: {label} - {class_name}")

## Generate from the trained outputs

This cell loads the trained angles/basis from the checkpoint through `generate_result`. It does not update model weights and does not save an image file.

In [ ]:
# Quick display test. If this checkerboard appears, notebook image output works.
test_pattern = (((__import__("numpy").indices((8, 8)).sum(axis=0) % 2) * 255).astype("uint8"))
display(Image.fromarray(test_pattern, mode="L").resize((160, 160), Image.Resampling.NEAREST))

In [ ]:
result = generate_result(
    label,
    shots=shots,
    seed=seed,
    prompt=prompt,
    variation=variation,
    latent_scale=latent_scale,
    candidates=candidates,
    backend="simulator",
)

image = result.image
print("Backend:", result.quantum_backend)
print("Seed used:", result.seed)

display_image = Image.fromarray((image * 255).astype("uint8"), mode="L")
display_image = display_image.resize((288, 288), Image.Resampling.NEAREST)

display(display_image)

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
plt.title(f"{prompt} -> {class_name}")
plt.axis("off")
plt.show()

## Available classes

In [ ]:
for index, name in enumerate(FASHION_CLASSES):
    print(f"{index}: {name}")